# Introduction to Data Science Lab — Practical Notebook

**Experiments included:** Questions 1–25 and 37–40 from the supplied *Introduction to Data Science Lab* manual (B. Tech 5th Semester).

This notebook follows the programs and datasets used in the manual. Each experiment has its own objective, concept notes, code cell, and output guide. The code is kept faithful to the manual; only line wrapping/indentation and the formatting of a split import statement have been adjusted so that it runs as valid Python in a Jupyter notebook.

## Using this notebook in VS Code

1. Open this `.ipynb` file in Visual Studio Code.
2. Install the **Python** and **Jupyter** extensions if VS Code prompts you to do so.
3. Select a Python 3 kernel from the kernel picker at the top-right.
4. Run the setup cell below once in that environment.
5. Run the experiments from top to bottom, or run an individual experiment's code cell after its dependencies are installed.

**Dataset note:** The Iris and Titanic datasets loaded through Seaborn are fetched by Seaborn when needed. An internet connection may be required the first time those dataset cells are executed.

## Experiment index

- Q1–Q5: Data handling and descriptive statistics
- Q6–Q10: Statistical testing and visualization
- Q11–Q15: Regression and model evaluation
- Q16–Q22: Classification algorithms
- Q23–Q25: Clustering
- Q37–Q40: Interactive visualization, multidimensional data, EDA, and an end-to-end application

## 0. Install the required libraries

Run this cell once in the selected VS Code Python environment. It installs the packages used by the selected experiments. `openpyxl` is needed for Excel files, while `plotly` is needed for Q37. NLTK is not included because it is only used by Question 31, which is outside the requested set.

In [ ]:
%pip install pandas numpy matplotlib seaborn scipy scikit-learn openpyxl plotly

---

## Question 1. Import and export datasets in CSV, Excel, and JSON formats using Pandas

**Lab manual reference:** Pages 4.

### Aim
Create a small table, save it in three common file formats, load each file back into Pandas, and print the imported CSV data.

### Concept / theory
CSV stores tabular data as comma-separated text; Excel stores worksheets in a workbook; JSON stores records in a structured text format. Pandas provides `to_csv`, `to_excel`, and `to_json` for exporting, and matching `read_*` functions for importing.

### Code walkthrough
The DataFrame is first created from two lists. Each `to_*` method writes a file to the notebook's current working directory. The `index=False` argument avoids saving Pandas' row index as an extra column. The three `read_*` methods then load the files into separate DataFrames. The JSON `orient='records'` setting stores each row as a JSON object.

### Expected output and interpretation
The output displays the three rows read from `data.csv`. The files `data.csv`, `data.xlsx`, and `data.json` are created in the current working directory.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd
import numpy as np

# Create a sample DataFrame
df = pd.DataFrame({'Name': ['Alice', 'Bob', 'Charlie'], 'Age': [25, 30, 35]})

# Export datasets
df.to_csv('data.csv', index=False)
df.to_json('data.json', orient='records')
# Requires openpyxl: pip install openpyxl
df.to_excel('data.xlsx', index=False)

# Import datasets
df_csv = pd.read_csv('data.csv')
df_json = pd.read_json('data.json')
df_excel = pd.read_excel('data.xlsx')

print("CSV Data:\n", df_csv)
print()
print("JSON Data:\n", df_json)
print()
print("EXCEL Data:\n", df_excel)

CSV Data:
    Unnamed: 0     Name  Age
0           0    Alice   25
1           1      Bob   30
2           2  Charlie   35

JSON Data:
       Name  Age
0    Alice   25
1      Bob   30
2  Charlie   35

EXCEL Data:
    Unnamed: 0     Name  Age
0           0    Alice   25
1           1      Bob   30
2           2  Charlie   35


---

## Question 2. Data preprocessing: missing values, duplicate records, and inconsistent data

**Lab manual reference:** Pages 4–5.

### Aim
Clean a sample employee dataset by filling missing ages, removing duplicate rows, standardizing names, and replacing an invalid negative salary.

### Concept / theory
Data preprocessing makes data more consistent before analysis. Missing values can be imputed, duplicates can be removed, and inconsistent text or invalid numeric values can be corrected or replaced.

### Code walkthrough
The `Age` column's missing entries are filled with its median. `drop_duplicates()` removes repeated rows. `str.strip()` removes leading/trailing spaces, and `str.title()` standardizes name capitalization. The `apply` function changes negative salaries to `NaN`; those missing/invalid salaries are then filled using the median salary.

### Expected output and interpretation
The printed DataFrame contains cleaned names, no duplicate Bob row, no missing ages, and no negative salary. The negative salary is replaced with the median of the remaining valid salary values.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd
import numpy as np

data = {'Name': ['Alice', 'Bob', 'Bob', 'David', 'eve '],
        'Age': [25, np.nan, np.nan, 40, 22],
        'Salary': [50000, 60000, 60000, -10000, 45000]}
df = pd.DataFrame(data)

# 1. Handle missing values (Fill with median)
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Handle duplicate records
df = df.drop_duplicates()

# 3. Handle inconsistent data (Strip whitespace, fix negative salary)
df['Name'] = df['Name'].str.strip().str.title()
df['Salary'] = df['Salary'].apply(lambda x: x if x > 0 else np.nan)
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

print(df)

---

## Question 3. Descriptive statistical analysis

**Lab manual reference:** Page 5.

### Aim
Calculate mean, median, mode, variance, standard deviation, and covariance for two numerical columns.

### Concept / theory
Descriptive statistics summarize the central value and spread of data. Mean is the arithmetic average; median is the middle value; mode is the most frequent value; variance measures spread in squared units; standard deviation is the square root of variance; covariance indicates whether two variables tend to move together.

### Code walkthrough
The two columns `A` and `B` are stored in a DataFrame. Pandas calculates column-wise statistics using `mean()`, `median()`, `mode()`, `var()`, and `std()`. `mode().iloc[0]` selects the first mode row. `cov()` returns the covariance matrix between columns.

### Expected output and interpretation
The notebook prints a statistic for each column and a 2×2 covariance matrix. Since B is consistently five units higher than A, their covariance is positive.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'A': [10, 20, 20, 30, 40, 50],
    'B': [15, 25, 25, 35, 45, 55]
})

print("Mean:\n", df.mean())
print("\nMedian:\n", df.median())
print("\nMode:\n", df.mode().iloc[0])
print("\nVariance:\n", df.var())
print("\nStandard Deviation:\n", df.std())
print("\nCovariance Matrix:\n", df.cov())

---

## Question 4. Covariance and correlation matrices with heatmaps

**Lab manual reference:** Page 6.

### Aim
Generate a multivariate dataset, calculate covariance and correlation matrices, and visualize both matrices as heatmaps.

### Concept / theory
Covariance shows the direction of joint variation and depends on the variables' scales. Correlation is standardized covariance and ranges from −1 to +1. A heatmap uses colour and optional cell annotations to make matrix values easier to compare.

### Code walkthrough
A fixed random seed makes the generated values repeatable. `np.random.randn(100, 4)` creates 100 observations for four features. `df.cov()` and `df.corr()` calculate the matrices. `plt.subplots(1, 2)` creates side-by-side axes, and Seaborn's `heatmap` displays each matrix with numerical annotations.

### Expected output and interpretation
Two heatmaps appear: one for covariance and one for correlation. The correlation diagonal is 1 because every feature is perfectly correlated with itself. Off-diagonal values show pairwise relationships.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Generate multivariate data
np.random.seed(42)
data = np.random.randn(100, 4)
df = pd.DataFrame(data, columns=['F1', 'F2', 'F3', 'F4'])

cov_matrix = df.cov()
corr_matrix = df.corr()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cov_matrix, annot=True, cmap='coolwarm', ax=ax[0])
ax[0].set_title('Covariance Matrix')

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', ax=ax[1])
ax[1].set_title('Correlation Matrix')

plt.show()

---

## Question 5. Univariate and multivariate normal distributions

**Lab manual reference:** Pages 6–7.

### Aim
Generate and visualize one normally distributed variable and a two-variable normal distribution.

### Concept / theory
A univariate normal distribution describes one variable and is shaped by its mean and standard deviation. A multivariate normal distribution describes multiple variables together and also includes covariance, which controls how the variables vary together.

### Code walkthrough
`np.random.normal(loc=0, scale=1, size=1000)` generates 1,000 values from a normal distribution with mean 0 and standard deviation 1. `np.random.multivariate_normal` generates 1,000 two-dimensional observations using the supplied mean vector and covariance matrix. The first plot uses a histogram with a KDE curve; the second uses a two-dimensional KDE plot.

### Expected output and interpretation
The first graph should look approximately bell-shaped. The second graph shows the joint density of the two variables. Because the covariance is positive (0.8), the two variables tend to increase together.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Univariate Normal
uni_data = np.random.normal(loc=0, scale=1, size=1000)

# Multivariate Normal
mean = [0, 0]
cov = [[1, 0.8], [0.8, 1]]
multi_data = np.random.multivariate_normal(mean, cov, size=1000)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(uni_data, kde=True, ax=ax[0])
ax[0].set_title('Univariate Normal Distribution')

sns.kdeplot(x=multi_data[:, 0], y=multi_data[:, 1], fill=True, ax=ax[1])
ax[1].set_title('Multivariate Normal Distribution')

plt.show()

---

## Question 6. Hypothesis testing using SciPy

**Lab manual reference:** Page 7.

### Aim
Use an independent two-sample t-test to check whether two generated groups have statistically different means.

### Concept / theory
Hypothesis testing uses sample data to assess a claim about a population. The null hypothesis here is that the two groups have equal population means. The alternative is that their means differ. The p-value measures how surprising the observed result would be if the null hypothesis were true.

### Code walkthrough
Two samples are generated from normal distributions with different stated means. `stats.ttest_ind` performs an independent samples t-test. The code compares the p-value with 0.05: a value below 0.05 leads to rejection of the null hypothesis; otherwise the code fails to reject it. The samples are randomly generated, so the exact statistic and p-value can vary between runs.

### Expected output and interpretation
The output prints a t-statistic and p-value, followed by the decision at the 5% significance level. 'Fail to reject' does not prove that the means are equal; it means the test did not find sufficient evidence of a difference.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
from scipy import stats

# Two independent samples
group_a = np.random.normal(50, 10, 30)
group_b = np.random.normal(55, 10, 30)

# Independent t-test
t_stat, p_value = stats.ttest_ind(group_a, group_b)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject the null hypothesis (Significant difference).")
else:
    print("Fail to reject the null hypothesis (No significant difference).")

---

## Question 7. Confidence interval for a population mean

**Lab manual reference:** Page 8.

### Aim
Calculate a 95% confidence interval for the population mean using a sample and the t-distribution.

### Concept / theory
A confidence interval is a range of plausible values for a population parameter. The standard error estimates the variability of the sample mean. When the population standard deviation is unknown, a t-distribution is commonly used to form a mean confidence interval.

### Code walkthrough
The code generates 50 observations, calculates their sample mean, and obtains the standard error using `stats.sem`. `stats.t.interval` calculates the interval using 49 degrees of freedom (`len(data)-1`), the sample mean as the centre, and the standard error as the scale.

### Expected output and interpretation
The output displays the sample mean and lower/upper confidence limits. In repeated sampling, a 95% confidence-interval procedure is expected to capture the true population mean in about 95% of samples, under its assumptions.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
import scipy.stats as stats

data = np.random.normal(loc=100, scale=15, size=50)
confidence = 0.95

mean = np.mean(data)
sem = stats.sem(data)  # Standard error of the mean

# Compute CI using t-distribution
ci = stats.t.interval(confidence, len(data) - 1, loc=mean, scale=sem)

print(f"Sample Mean: {mean:.2f}")
print(f"95% Confidence Interval: ({ci[0]:.2f}, {ci[1]:.2f})")

---

## Question 8. Dataset manipulation and transformation with Pandas

**Lab manual reference:** Pages 8–9.

### Aim
Add a bonus column, calculate average salary department-wise, and create a pivot table.

### Concept / theory
Data manipulation means changing or summarizing a dataset to make it useful for analysis. A derived column is calculated from existing columns. `groupby` aggregates records by category, while a pivot table summarizes data across selected dimensions.

### Code walkthrough
The DataFrame contains department, employee, and salary. The `Bonus` column is calculated as 10% of salary. `groupby('Department')['Salary'].mean()` finds the mean salary for each department. `pivot_table` computes both the mean and maximum salary for every department.

### Expected output and interpretation
The first printed table shows average salary by department. The pivot table shows mean and maximum salary for HR, IT, and Finance.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Department': ['IT', 'HR', 'IT', 'Finance', 'HR'],
    'Employee': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Salary': [70000, 50000, 80000, 90000, 55000]
})

# 1. Add a new column (Transformation)
df['Bonus'] = df['Salary'] * 0.10

# 2. GroupBy manipulation
dept_summary = df.groupby('Department')['Salary'].mean().reset_index()

# 3. Pivot Table
pivot = df.pivot_table(values='Salary', index='Department', aggfunc=['mean', 'max'])

print("Grouped Summary:\n", dept_summary)
print("\nPivot Table:\n", pivot)

---

## Question 9. Data visualization using Matplotlib and Seaborn

**Lab manual reference:** Pages 9–10.

### Aim
Plot a sine wave using Matplotlib and Seaborn.

### Concept / theory
Data visualization presents numerical information graphically so that trends and patterns are easier to understand. A line plot is useful for showing how a continuous variable changes across an ordered axis.

### Code walkthrough
`np.linspace(0, 10, 100)` creates 100 evenly spaced x-values. `np.sin(x)` calculates the sine for each value. `sns.lineplot` draws the line, while the remaining Matplotlib functions add the title, axis labels, legend, grid, and display the figure.

### Expected output and interpretation
A smooth sine wave is displayed from x = 0 to x = 10. The y-values oscillate between approximately −1 and +1.

### Program
Run the following cell to execute the program.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

x = np.linspace(0, 10, 100)
y = np.sin(x)

plt.figure(figsize=(8, 4))
sns.lineplot(x=x, y=y, color='red', label='Sine Wave')
plt.title('Basic Line Plot using Seaborn/Matplotlib')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.grid(True)
plt.show()

---

## Question 10. Bar chart, histogram, pie chart, box plot, scatter plot, and heatmap

**Lab manual reference:** Pages 10–11.

### Aim
Create six commonly used visualizations in a single figure using Matplotlib and Seaborn.

### Concept / theory
Different charts answer different questions. Bar charts compare categories; histograms show a numerical distribution; pie charts show category proportions; box plots summarize spread and potential outliers; scatter plots show relationships between two numeric variables; heatmaps show values in a matrix.

### Code walkthrough
The code prepares random numerical data and three category values, then creates a 2×3 grid of plots. Each plotting function draws one chart on its assigned axis. `tight_layout()` adjusts spacing to reduce overlap, and `show()` renders the figure.

### Expected output and interpretation
A figure containing six panels appears. The generated random data means the histogram, box plot, scatter plot, and heatmap may look different each time. The bar and pie charts use fixed category values A, B, and C.

### Program
Run the following cell to execute the program.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

data = np.random.randn(100)
categories = ['A', 'B', 'C']
values = [10, 20, 15]

fig, axs = plt.subplots(2, 3, figsize=(15, 10))

# Bar Chart
axs[0, 0].bar(categories, values, color='skyblue')
axs[0, 0].set_title('Bar Chart')

# Histogram
axs[0, 1].hist(data, bins=15, color='green', alpha=0.7)
axs[0, 1].set_title('Histogram')

# Pie Chart
axs[0, 2].pie(values, labels=categories, autopct='%1.1f%%')
axs[0, 2].set_title('Pie Chart')

# Box Plot
sns.boxplot(y=data, ax=axs[1, 0])
axs[1, 0].set_title('Box Plot')

# Scatter Plot
sns.scatterplot(x=np.random.rand(50), y=np.random.rand(50), ax=axs[1, 1])
axs[1, 1].set_title('Scatter Plot')

# Heatmap
sns.heatmap(np.random.rand(5, 5), cmap='viridis', ax=axs[1, 2])
axs[1, 2].set_title('Heatmap')

plt.tight_layout()
plt.show()

---

## Question 11. Simple Linear Regression using Scikit-learn

**Lab manual reference:** Pages 11–12.

### Aim
Fit a simple linear regression model using one input feature and visualize the fitted regression line.

### Concept / theory
Simple linear regression models the relationship between one independent variable and one dependent variable using a straight line: y = mx + c. The model learns the slope (coefficient) and intercept that best fit the training observations by least squares.

### Code walkthrough
`X` contains one feature and is represented as a two-dimensional array, as expected by Scikit-learn. `y` contains target values. `LinearRegression()` creates the model, `fit()` learns its parameters, and `predict()` calculates fitted values. The scatter plot shows observations and the line plot shows model predictions. Finally, the learned coefficient and intercept are printed.

### Expected output and interpretation
The graph shows five data points and the fitted regression line. The printed coefficient is the slope and the intercept is the predicted y-value when x is zero.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Generate sample data
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([2, 4, 5, 4, 5])

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

plt.scatter(X, y, color='blue', label='Data')
plt.plot(X, y_pred, color='red', label='Regression Line')
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

print(f"Coefficient: {model.coef_[0]:.2f}, Intercept: {model.intercept_:.2f}")

---

## Question 12. Multiple Linear Regression and model evaluation

**Lab manual reference:** Page 12.

### Aim
Train a multiple linear regression model using three input features and evaluate its predictions with MSE and R².

### Concept / theory
Multiple linear regression predicts a continuous target from two or more input features. Mean Squared Error (MSE) is the average squared prediction error, where lower is better. R² measures the proportion of target variation explained by the model; values closer to 1 generally indicate a better fit.

### Code walkthrough
The code creates 100 rows with three random features and constructs a target from a linear combination of those features plus noise. `train_test_split` reserves 20% of the observations for testing. The model is fitted on the training portion, predicts the test portion, and reports MSE, R², and learned coefficients.

### Expected output and interpretation
MSE and R² describe performance on the held-out test data. Three coefficients correspond to the three input features. Exact values can vary because the original train-test split does not specify a `random_state`.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Generate data with 3 features
np.random.seed(42)
X = np.random.rand(100, 3)
y = 3 * X[:, 0] + 5 * X[:, 1] - 2 * X[:, 2] + np.random.randn(100) * 0.1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))
print("Coefficients:", model.coef_)

---

## Question 13. Ridge Regression and Lasso Regression

**Lab manual reference:** Pages 12–13.

### Aim
Fit Ridge and Lasso regression models and compare their R² scores and coefficients.

### Concept / theory
Ridge and Lasso are regularized linear regression methods. Both add a penalty to discourage overly large coefficients. Ridge uses an L2 penalty and usually shrinks coefficients toward zero. Lasso uses an L1 penalty and can set some coefficients exactly to zero, which can perform feature selection. `alpha` controls penalty strength.

### Code walkthrough
The data contains five input features and a target mostly determined by the first two features plus random noise. The data is split into training and testing sets. A Ridge model with `alpha=1.0` and a Lasso model with `alpha=0.1` are each trained. Their test R² scores and coefficient arrays are printed.

### Expected output and interpretation
Compare the two R² values to see how well each model predicts the held-out target. In the Lasso coefficient list, some values may be exactly zero, indicating that the corresponding features were excluded by the model. Exact values can vary because the feature matrix is not seeded in this original program.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

X = np.random.rand(100, 5)
y = 2 * X[:, 0] + 3 * X[:, 1] + np.random.randn(100) * 0.5

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
print("Ridge Score (R2):", ridge.score(X_test, y_test))
print("Ridge Coefs:", ridge.coef_)

# Lasso Regression
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
print("\nLasso Score (R2):", lasso.score(X_test, y_test))
print("Lasso Coefs:", lasso.coef_)  # Notice sparsity (zeros)

---

## Question 14. Model selection using train-test split and cross-validation

**Lab manual reference:** Pages 13–14.

### Aim
Compare a single train-test split score with a 5-fold cross-validation score using Logistic Regression on Iris.

### Concept / theory
A train-test split evaluates a model on one held-out subset. Cross-validation divides the dataset into several folds and repeats training/testing so the estimate is less dependent on a single split. In 5-fold cross-validation, each fold is used once for validation while the other four folds are used for training.

### Code walkthrough
`load_iris(return_X_y=True)` loads the features and labels. A Logistic Regression model is created. The train-test method trains on 70% and evaluates on 30%. `cross_val_score` then calculates five validation scores; the code prints their mean and twice their standard deviation as a compact measure of variation.

### Expected output and interpretation
The output gives two accuracy estimates. They may differ because the methods use different partitions. The 5-fold result is reported as mean accuracy plus/minus twice the standard deviation.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=200)

# 1. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model.fit(X_train, y_train)
split_acc = model.score(X_test, y_test)
print(f"Train-Test Split Accuracy: {split_acc:.4f}")

# 2. Cross-Validation
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

---

## Question 15. Model evaluation using confusion matrix and classification metrics

**Lab manual reference:** Pages 14–15.

### Aim
Train a binary Logistic Regression classifier and calculate confusion matrix, accuracy, precision, recall, F1-score, and ROC-AUC.

### Concept / theory
A confusion matrix counts true positives, true negatives, false positives, and false negatives. Accuracy is the fraction of correct predictions. Precision measures how many predicted positives are correct. Recall measures how many actual positives are found. F1-score is the harmonic mean of precision and recall. ROC-AUC measures how well prediction scores separate the two classes across thresholds.

### Code walkthrough
`make_classification` creates a synthetic two-class dataset. The data is split into train and test sets, and Logistic Regression is fitted. `predict` produces class labels for the confusion matrix and threshold-based metrics. `predict_proba(... )[:, 1]` obtains the estimated probability of the positive class for ROC-AUC. The import statement is formatted across lines in the notebook for valid Python syntax.

### Expected output and interpretation
The output shows the confusion matrix and five evaluation metrics. The exact values may vary because the source program does not set a random state in `train_test_split`. ROC-AUC uses probabilities rather than the predicted class labels.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

X, y = make_classification(n_samples=200, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

model = LogisticRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

---

## Question 16. k-Nearest Neighbors (kNN) classification

**Lab manual reference:** Page 15.

### Aim
Train a kNN classifier on the Iris dataset and calculate its test accuracy.

### Concept / theory
k-Nearest Neighbors is a supervised classification algorithm. To classify a new point, it identifies the k closest training points and assigns the most common class among those neighbors. Here, `k=5`.

### Code walkthrough
The Iris dataset provides four numeric features and a species label. `train_test_split` allocates 20% for testing. `KNeighborsClassifier(n_neighbors=5)` creates the classifier. `fit` stores the training examples, and `predict` labels the unseen test examples. `accuracy_score` compares predictions with actual labels.

### Expected output and interpretation
The output is one accuracy value between 0 and 1. It represents the proportion of test examples classified correctly.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print("kNN Accuracy:", accuracy_score(y_test, y_pred))

---

## Question 17. Decision Tree classification

**Lab manual reference:** Pages 15–16.

### Aim
Train a Decision Tree classifier on the Iris dataset and report test accuracy.

### Concept / theory
A Decision Tree classifies data by repeatedly splitting it according to feature values. The `gini` criterion measures class impurity. `max_depth=3` limits the tree to three levels, helping constrain its complexity.

### Code walkthrough
The Iris features and labels are loaded and split into training and testing sets. The classifier is initialized with the Gini criterion and maximum depth of 3. It learns decision rules from the training set, predicts labels for the test set, and prints accuracy.

### Expected output and interpretation
The output is the percentage (as a fraction from 0 to 1) of the test observations classified correctly. The depth setting restricts how complex the learned tree can become.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(criterion='gini', max_depth=3)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))

---

## Question 18. Support Vector Machine (SVM) classification

**Lab manual reference:** Page 16.

### Aim
Use a linear Support Vector Machine to classify the breast cancer dataset.

### Concept / theory
A Support Vector Machine finds a decision boundary that separates classes while trying to maximize the margin between them. A linear kernel creates a linear decision boundary in the feature space. Support vectors are the training points closest to that boundary.

### Code walkthrough
The breast cancer dataset is loaded and divided into training and testing parts. `SVC(kernel='linear')` creates an SVM with a linear kernel. The model is fitted, test labels are predicted, and accuracy is calculated.

### Expected output and interpretation
The output is the test accuracy. This manual example does not scale the features; for many real datasets, feature scaling can materially affect SVM performance, but the code here follows the supplied manual.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred))

---

## Question 19. Random Forest classification

**Lab manual reference:** Pages 16–17.

### Aim
Train a Random Forest classifier on the Wine dataset and report test accuracy.

### Concept / theory
A Random Forest is an ensemble of decision trees. Each tree is trained using a sampled dataset and feature subsets; their predictions are combined to produce the final class. Using many trees can make predictions more stable than relying on one decision tree.

### Code walkthrough
The Wine dataset contains numerical chemical measurements and class labels. After the train-test split, `RandomForestClassifier(n_estimators=100)` creates a forest of 100 trees. `random_state=42` makes the model's random choices reproducible. The classifier is trained, predicts the test set, and its accuracy is printed.

### Expected output and interpretation
The output is the fraction of test records assigned to the correct wine class. `n_estimators=100` means the ensemble contains 100 decision trees.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))

---

## Question 20. Naïve Bayes classification

**Lab manual reference:** Page 17.

### Aim
Train a Gaussian Naïve Bayes classifier on the Iris dataset and calculate accuracy.

### Concept / theory
Naïve Bayes is a probabilistic classifier based on Bayes' theorem. It assumes that features are conditionally independent given the class. Gaussian Naïve Bayes models each feature within each class using a normal distribution.

### Code walkthrough
The Iris features and labels are split into training and testing sets. `GaussianNB()` creates the classifier. The model estimates class probabilities from the training data, predicts the test labels, and reports accuracy.

### Expected output and interpretation
The printed accuracy indicates the proportion of test records classified correctly. The algorithm is called 'naïve' because of its conditional-independence assumption.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred = nb.predict(X_test)
print("Naïve Bayes Accuracy:", accuracy_score(y_test, y_pred))

---

## Question 21. Logistic Regression for binary classification

**Lab manual reference:** Pages 17–18.

### Aim
Train a Logistic Regression model on a synthetic two-class dataset and calculate accuracy.

### Concept / theory
Logistic Regression is a classification algorithm despite its name. It estimates the probability of a class using the logistic (sigmoid) function and assigns a class based on a decision threshold, commonly 0.5 for binary classification.

### Code walkthrough
`make_classification` creates 500 synthetic samples with two classes. The data is divided into 80% training and 20% testing. `LogisticRegression()` learns the relationship between the features and binary target. The model predicts test labels, and `accuracy_score` evaluates them.

### Expected output and interpretation
The output is the model's test accuracy. Since `random_state=42` is used both for data generation and splitting, the result should be repeatable for the same compatible library version.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))

---

## Question 22. Compare classification algorithms on the same dataset

**Lab manual reference:** Page 18.

### Aim
Compare Logistic Regression, Decision Tree, SVM, and kNN using the same train-test split of the breast cancer dataset.

### Concept / theory
A fair model comparison keeps the dataset and train-test split consistent so that differences in performance are mainly due to the algorithms. Accuracy is a simple metric for this comparison, although other metrics may be needed for imbalanced datasets or when error costs differ.

### Code walkthrough
The breast cancer dataset is split once. The four classifiers are stored in a dictionary. The loop trains each model on the same training subset, predicts the same test subset, calculates accuracy, and prints the result with four decimal places. Logistic Regression uses a larger iteration limit to help it converge.

### Expected output and interpretation
Four accuracy values are printed, one for each algorithm. The highest accuracy is the best performer under this particular split and metric; this does not automatically mean it is best for every use case.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name} Accuracy: {acc:.4f}")

---

## Question 23. K-Means clustering

**Lab manual reference:** Pages 19–20.

### Aim
Apply K-Means to a generated dataset containing four clusters and plot the cluster assignments and centroids.

### Concept / theory
K-Means is an unsupervised clustering algorithm. It assigns each observation to the nearest centroid, then updates each centroid to the mean of the observations assigned to it. This process repeats until the assignments or centroids stabilize. The number of clusters, k, must be chosen in advance.

### Code walkthrough
`make_blobs` generates 300 two-dimensional observations around four centres. `KMeans(n_clusters=4)` requests four clusters. `fit_predict` fits the algorithm and returns a cluster label for every observation. The first scatter plot colours points by assigned cluster; the second marks the learned centroids with red X markers.

### Expected output and interpretation
The graph should show four coloured groups and four red X markers. Cluster labels such as 0, 1, 2, and 3 are arbitrary identifiers, not rankings.

### Program
Run the following cell to execute the program.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# Generate random cluster data
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

kmeans = KMeans(n_clusters=4, random_state=42)
y_kmeans = kmeans.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, s=50, cmap='viridis')
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.75, marker='X')
plt.title("K-Means Clustering")
plt.show()

---

## Question 24. Hierarchical clustering and dendrogram

**Lab manual reference:** Pages 19–20.

### Aim
Perform agglomerative hierarchical clustering using Ward linkage and display a dendrogram.

### Concept / theory
Hierarchical clustering builds a hierarchy of clusters. In agglomerative clustering, each observation begins as its own cluster and clusters are repeatedly merged. A dendrogram is a tree diagram showing the order and distance at which clusters are joined. Ward linkage merges clusters to minimize the increase in within-cluster variance.

### Code walkthrough
`make_blobs` creates 50 two-dimensional observations. `linkage(X, method='ward')` calculates the hierarchical merge structure. `dendrogram(Z)` visualizes that structure. The x-axis shows sample indices, and the y-axis shows the linkage distance at which merges occur.

### Expected output and interpretation
The dendrogram branches show how individual samples and groups are merged. A large vertical gap between merge heights can suggest a useful place to cut the tree into clusters.

### Program
Run the following cell to execute the program.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=50, centers=3, random_state=42)

# Perform hierarchical clustering
Z = linkage(X, method='ward')

plt.figure(figsize=(10, 5))
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample index')
plt.ylabel('Distance')
dendrogram(Z)
plt.show()

---

## Question 25. Compare clustering algorithms using silhouette score

**Lab manual reference:** Pages 20–21.

### Aim
Compare K-Means and Agglomerative (single-linkage) clustering on a two-moons dataset using silhouette scores.

### Concept / theory
Clustering is unsupervised, so there may be no known labels for direct accuracy calculation. The silhouette score measures how similar a point is to its own cluster compared with other clusters. Scores range from −1 to +1; larger values generally indicate better-separated, more cohesive clusters.

### Code walkthrough
`make_moons` generates a curved two-cluster dataset. K-Means assigns two clusters based on centroids. Agglomerative clustering also creates two clusters but uses single linkage. `silhouette_score` evaluates each set of cluster labels against the original feature data. The two scores are printed for comparison.

### Expected output and interpretation
The output contains one silhouette score per algorithm. On curved two-moons data, the algorithms can behave differently because K-Means favours roughly compact groups while single linkage can follow connected shapes. A higher score is better according to this metric, but visual inspection and domain knowledge are also useful.

### Program
Run the following cell to execute the program.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

X, _ = make_moons(n_samples=200, noise=0.05, random_state=42)

# KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
labels_km = kmeans.fit_predict(X)
score_km = silhouette_score(X, labels_km)

# Hierarchical (Agglomerative)
agglo = AgglomerativeClustering(n_clusters=2, linkage='single')
labels_agg = agglo.fit_predict(X)
score_agg = silhouette_score(X, labels_agg)

print(f"K-Means Silhouette Score: {score_km:.4f}")
print(f"Agglomerative Silhouette Score: {score_agg:.4f}")

---

## Question 37. Interactive data visualization using Plotly

**Lab manual reference:** Page 28.

### Aim
Create an interactive scatter plot of the Iris dataset using Plotly Express.

### Concept / theory
Plotly creates interactive charts that support actions such as hovering, zooming, and panning. A scatter plot represents each observation as a point. Colour can encode a category, while marker size can encode another numeric variable.

### Code walkthrough
`px.data.iris()` loads Plotly's sample Iris dataset. `px.scatter` maps sepal width to the x-axis and sepal length to the y-axis. Points are coloured by species, sized by petal length, and show petal width in hover details. `fig.show()` renders the interactive chart in the notebook or browser depending on the VS Code/Jupyter renderer configuration.

### Expected output and interpretation
An interactive scatter plot appears. Hover over a point to inspect its petal width and other hover information; use the toolbar to zoom, pan, or reset the plot. The point colour distinguishes the three Iris species.

### Program
Run the following cell to execute the program.

In [ ]:
# Requires: pip install plotly pandas
import plotly.express as px
import pandas as pd

df = px.data.iris()

fig = px.scatter(
    df,
    x="sepal_width",
    y="sepal_length",
    color="species",
    size="petal_length",
    hover_data=["petal_width"],
    title="Interactive Iris Scatter Plot"
)

# To show the plot in a browser/notebook:
fig.show()

---

## Question 38. Multidimensional datasets using pair plots and dimensionality reduction

**Lab manual reference:** Pages 28–29.

### Aim
Use a pair plot to inspect pairwise feature relationships and PCA to project the Iris features into two dimensions.

### Concept / theory
A multidimensional dataset has several features for each observation. A pair plot shows pairwise relationships between numerical features, often with distributions along the diagonal. Principal Component Analysis (PCA) transforms correlated features into new axes called principal components, ordered by the amount of variance they explain. Reducing to two components makes a high-dimensional dataset easier to visualize.

### Code walkthrough
Seaborn loads the Iris dataset. `pairplot(df, hue='species')` produces pairwise plots and colours observations by species. PCA is then configured with two components and fitted to the four numeric columns (`df.iloc[:, :-1]`, excluding the final species column). The reduced values are placed in a new DataFrame, the species labels are attached, and a 2D scatter plot is drawn.

### Expected output and interpretation
The first output is a grid of pairwise scatter plots and feature distributions. The second output is a two-dimensional PCA scatter plot. Points closer together have more similar projections; species separation indicates how well the first two principal components distinguish the groups.

### Program
Run the following cell to execute the program.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd

df = sns.load_dataset('iris')

# 1. Pair Plot
sns.pairplot(df, hue='species')
plt.suptitle("Pair Plot of Iris Features", y=1.02)
plt.show()

# 2. Dimensionality Reduction Plot
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(df.iloc[:, :-1])
df_pca = pd.DataFrame(reduced_data, columns=['PC1', 'PC2'])
df_pca['species'] = df['species']

sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='species')
plt.title("2D PCA Visualization")
plt.show()

---

## Question 39. Exploratory Data Analysis (EDA) on a real-world dataset

**Lab manual reference:** Pages 29–30.

### Aim
Perform basic EDA on the Titanic dataset by inspecting its structure, summary statistics, missing values, and survival counts by ticket class.

### Concept / theory
Exploratory Data Analysis is the process of examining data before formal modelling. It helps identify the dataset's size and types, numerical patterns, missing values, outliers, and relationships. `info()` gives structural information, `describe()` summarizes numeric columns, and `isnull().sum()` counts missing entries per column.

### Code walkthrough
Seaborn loads its Titanic sample dataset. The code prints DataFrame information, descriptive statistics, and missing-value counts. Finally, `countplot` compares the number of passengers who survived or did not survive within each ticket class. The `hue='survived'` argument separates bars by survival outcome.

### Expected output and interpretation
The cell prints the dataset structure and summary tables, followed by a count plot. The chart helps compare survival counts among First, Second, and Third class passengers. The plot shows counts, not survival percentages.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load titanic dataset
df = sns.load_dataset('titanic')

print("--- DataFrame Info ---")
df.info()

print("\n--- Summary Statistics ---")
print(df.describe())

print("\n--- Missing Values ---")
print(df.isnull().sum())

# Plot survival by class
sns.countplot(data=df, x='class', hue='survived')
plt.title('Survival Count by Ticket Class')
plt.show()

---

## Question 40. End-to-end data science application on a real-world dataset

**Lab manual reference:** Pages 30–31.

### Aim
Build a basic end-to-end classification workflow on the Titanic dataset: load, preprocess, visualize, train a model, evaluate it, and make a prediction.

### Concept / theory
An end-to-end data science workflow connects the stages of a practical project. Preprocessing handles missing and categorical data. Visualization helps inspect feature relationships. Model development fits an algorithm to prepared features and a target. Evaluation measures performance on unseen test data. Prediction applies the trained model to a new or held-out example.

### Code walkthrough
The code loads Titanic data and removes columns considered irrelevant or to have too many missing values. Missing ages are filled with the median, and rows with remaining missing data are dropped. `get_dummies` converts the selected categorical columns into numeric indicator columns. A correlation heatmap visualizes numeric feature relationships. The target column `survived` is separated from the features. A Random Forest is trained on 80% of the data and evaluated on 20% using a confusion matrix and classification report. Finally, the first test passenger is passed to `predict`.

### Expected output and interpretation
The output includes a feature-correlation heatmap, confusion matrix, classification report (precision, recall, F1-score, and support), and one prediction. The prediction is 1 for survived and 0 for did not survive. It is a demonstration of the workflow, not a guarantee that this model is suitable for real deployment; the manual's preprocessing and feature choices are retained.

### Program
Run the following cell to execute the program.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load Data
df = sns.load_dataset('titanic')

# 2. Preprocessing
# Drop columns with too many nulls or irrelevant features
df.drop(['deck', 'embark_town', 'alive', 'who', 'adult_male'], axis=1, inplace=True)
# Fill missing age with median
df['age'] = df['age'].fillna(df['age'].median())
# Drop remaining missing values
df.dropna(inplace=True)
# Convert categoricals
df = pd.get_dummies(df, columns=['sex', 'embarked', 'class'], drop_first=True)

# 3. Visualization
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()

# 4. Model Development
X = df.drop('survived', axis=1)
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluation
y_pred = model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 6. Prediction
sample_passenger = X_test.iloc[0:1]
pred = model.predict(sample_passenger)
print("\nPrediction for first passenger in test set (1=Survived, 0=Died):", pred[0])

---

## Final revision notes

- **Pandas** is used for DataFrames, file input/output, cleaning, grouping, and summary tables.
- **NumPy** is used for numerical arrays and random sample generation.
- **Matplotlib and Seaborn** are used for static plots and statistical visualizations.
- **SciPy** is used for statistical tests and confidence intervals.
- **Scikit-learn** is used for regression, classification, clustering, dimensionality reduction, and evaluation metrics.
- **Plotly** is used for interactive visualization.
- In supervised learning, the model learns from labelled examples. In unsupervised learning, it finds structure without target labels.
- Keep the test set separate from training. Use evaluation metrics appropriate to the task rather than relying on one score alone.

**Notebook source:** Programs are based on Questions 1–25 and 37–40 in the uploaded *Introduction to Data Science Lab* manual. Code cell outputs are intentionally empty so that you can execute the programs in your own VS Code environment and retain your own run results.